# Experiment 022: smooth 1000-step GRPO at 7B with the registered eval grid (Colab)

Three fresh GRPO arms from each seed's exact 013 merged base, one smooth 1000-step schedule, and the study's first prospectively fixed within-run trajectory:
checkpoints at 200, 400, 500 600,  800 and 1000, each evaluated generatively and by probe through hash-bound scratch copies.

Every arm completes regardless of intermediate results; only a
technical failure pauses rollout. Arms are resume-safe: on a disconnect
rerun all cells, finished stages skip. ~9h per arm including its grid, one
arm per session is realistic.

Hardware: A100-80GB required, same as every 7B run.

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [2]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
# cu_before = 221.7   # from the resources panel
cu_before = 156.8   # before the last seed running!
unit_rate = 6.77   # units per hour
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '022_colab_b7_smooth1000', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260731_231624 | cu_before = 156.8
Mounted at /content/drive
saved open session: 20260731_231624
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 80GB required

In [3]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime for the 7b runs'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-6bd45a98-2e1b-29d8-b1dc-9e0d9667e876)
total vram: 79.3 GiB


## 2. Clone or update the repo (automatic)

In [4]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently runing whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 2767, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 2767 (delta 150), reused 196 (delta 96), pack-reused 2516 (from 1)
Receiving objects: 100% (2767/2767), 80.47 MiB | 15.93 MiB/s, done.
Resolving deltas: 100% (1769/1769), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [5]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')

# pin every arm to the recorded 013 environment, so the
# reference manifest is required and a missing one stops the session
import importlib, json, os
ref = 'results/013_COLAB_FULL_B7_sft_grpo_seed0/manifest.json'
assert os.path.exists(ref), 'reference manifest missing, pull the repo with results before training'
par = json.load(open(ref))['versions']
for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
    have = importlib.import_module(lib).__version__
    assert have == par[lib], f'{lib} {have} differs from the recorded {par[lib]}'
print('environment matches the recorded 013 versions')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 149.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.6 MB/s eta 0:00:00
deps installed
environment matches the recorded 013 versions


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [6]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record and free space

In [7]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
# three merged 7b copies plus grid checkpoints need real room
assert free_gb >= 70, 'need >=70GB free on drive for the three E1 arms'

open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260731_231624.json
drive free: 179 GB


## 6. Arms and the registered grid

All three arms are authorised and bug-gated only. The grid is fixed and must not be edited after any training starts.

In [8]:
# SEEDS_TO_RUN = (0, 1, 2)
# run one seed first to check the pipeline
#SEEDS_TO_RUN = (0,)
# SEEDS_TO_RUN = (1,)

# let's run the last seed :)
SEEDS_TO_RUN = (2,)

GRID = (200, 400, 500, 600, 800, 1000)   # registered, step 0 is the recorded 013 sft baseline
print('arms this pass:', SEEDS_TO_RUN, '| grid:', GRID)

arms this pass: (2,) | grid: (200, 400, 500, 600, 800, 1000)


## 7. Seed the arm directories (merged base only, no checkpoint)

Fresh runs: each arm gets a hash-verified copy of its 013 counterpart's
merged base and a config hash written in the same transaction. A directory
with a manifest is verified for resume, a directory without one must be
empty.

In [9]:
import hashlib, json, os, shutil, glob, sys, time
sys.path.insert(0, os.getcwd())
from config import load_config
from checkpoint import write_config_hash, _config_hash
from dataclasses import asdict

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

D = '/content/drive/MyDrive/inm363'
for s in SEEDS_TO_RUN:
    SRC = os.path.join(D, 'checkpoints', f'013_COLAB_FULL_B7_sft_grpo_seed{s}')
    DST = os.path.join(D, 'checkpoints', f'022_smooth_B7_seed{s}')
    res = os.path.join(D, 'results', f'022_smooth_B7_seed{s}_step1000')
    man_path = os.path.join(res, 'arm_manifest.json')
    cfg = load_config(f'configs/022_smooth_colab_b7_seed{s}.yaml')
    assert cfg.experiment == '022_smooth' and cfg.train.seed == s, f'config identity mismatch for seed {s}'

    if os.path.exists(man_path):
        man = json.load(open(man_path))
        assert man['source_run'] == f'013_COLAB_FULL_B7_sft_grpo_seed{s}', f'seed {s}: manifest names a different source'
        for k, v in man['hashes'].items():
            f = os.path.join(DST, k)
            assert os.path.exists(f) and sha(f) == v, f'seed {s}: {k} drifted since seeding'
        hp = os.path.join(DST, 'grpo', 'config_hash.txt')
        assert os.path.exists(hp) and open(hp).read().strip() == _config_hash(cfg), f'seed {s}: config hash mismatch'
        # the core hash skips log and exec on purpose, so the grid cadence and
        # exec pins get their own binding here
        current = json.loads(json.dumps({'log': asdict(cfg.log), 'exec': asdict(cfg.exec)}, default=str))
        assert man.get('log_exec') == current, f'seed {s}: log or exec settings drifted since seeding'
        assert man.get('grid') == [0] + list(GRID), f'seed {s}: the registered grid changed since seeding'
        print(f'seed {s}: resume verified against the manifest and config hash')
        continue

    existing = glob.glob(os.path.join(DST, '*'))
    assert not existing, f'seed {s}: files present but no arm manifest, delete {DST} and rerun: {existing}'
    stale = [f for f in ('grpo_eval.json', 'grpo_preds.jsonl', 'grpo_logprob.json', 'grpo_logprob_preds.jsonl')
             if os.path.exists(os.path.join(res, f))]
    assert not stale, f'seed {s}: evaluation artifacts already present on a fresh arm: {stale}'

    hashes = {}
    src_dir, dst_dir = os.path.join(SRC, 'sft_merged'), os.path.join(DST, 'sft_merged')
    os.makedirs(dst_dir, exist_ok=True)
    for f in sorted(os.listdir(src_dir)):
        a, b = os.path.join(src_dir, f), os.path.join(dst_dir, f)
        if not os.path.exists(b):
            shutil.copy2(a, b)
        ha, hb = sha(a), sha(b)
        assert ha == hb, f'seed {s}: sft_merged/{f} copy hash differs'
        hashes[f'sft_merged/{f}'] = ha
    write_config_hash(os.path.join(DST, 'grpo'), cfg)
    written = open(os.path.join(DST, 'grpo', 'config_hash.txt')).read().strip()
    assert written == _config_hash(cfg), f'seed {s}: config hash write-back verification failed'
    os.makedirs(res, exist_ok=True)
    with open(man_path, 'w') as f:
        json.dump({'source_run': f'013_COLAB_FULL_B7_sft_grpo_seed{s}', 'seeded': 'sft_merged only, fresh run',
                   'ts': time.strftime('%Y-%m-%d %H:%M'), 'hashes': hashes,
                   'log_exec': json.loads(json.dumps({'log': asdict(cfg.log), 'exec': asdict(cfg.exec)}, default=str)),
                   'grid': [0] + list(GRID)}, f, indent=2)
    print(f'seed {s}: seeded and verified {len(hashes)} files')

seed 2: seeded and verified 6 files


## 8. The training arms

Fresh 1000-step runs, sentinel active, ladder off from the config. Each
attempt records its environment before training and closes in finally;
finished arms skip without new ledger entries.

**Failure rule:** a normal colab disconnect or deliberate cell stop can resume from the last verified checkpoint. An explicit worker failure or cuda oom blocks resume for review. The batch, accumulation and checkpointing settings never change automatically.

Training runs in a disposable process with live output, then releases its GPU memory before evaluation. A logging-only commit change is checked by its exact file diff and recorded in the arm manifest.


## 9. Completion, schedule and grid gates

completed arms are reverified here on every pass. The grid checkpoints must
all exist with matching trainer state before any evaluation is allowed.

In [ ]:
import glob as _glob, json, os, subprocess, sys, time
sys.path.insert(0, os.getcwd())
from config import load_config
from scripts.env_switch import resolve_device, apply_smoke
from train.model_utils import resolve_dtype
from obs import _git_info, _versions, gpu_snapshot, append_ledger
from scripts.grpo_worker import (active_worker_processes, blocked_resume_reason,
                                 stream_subprocess)

assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
D = '/content/drive/MyDrive/inm363'
if os.path.exists('wandb_api_key.txt'):
    os.environ.setdefault('WANDB_API_KEY', open('wandb_api_key.txt').read().strip())
os.environ.setdefault('WANDB_PROJECT', 'inm363')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

for s in SEEDS_TO_RUN:
    # each arm checks against its own parent's recorded environment
    parent = json.load(open(os.path.join(D, 'results', f'013_COLAB_FULL_B7_sft_grpo_seed{s}', 'manifest.json')))
    for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate', 'cais/mmlu@', 'openai/gsm8k@'):
        assert _versions().get(lib) == parent['versions'].get(lib), (
            f'seed {s}: {lib} differs from the parent run')
    config_path = f'configs/022_smooth_colab_b7_seed{s}.yaml'
    cfg = apply_smoke(load_config(config_path))
    from train.model_utils import resolve_dtype as _rd
    assert _rd(cfg) == parent['resolved_dtype'], f'seed {s}: dtype differs from the recorded parent'
    assert cfg.exec.allow_oom_fallback is False, 'the 022 configs must disable the ladder'
    resolve_device(cfg.env)
    out_dir = os.path.join(D, 'checkpoints', f'022_smooth_B7_seed{s}', 'grpo')
    res_dir = os.path.join(D, 'results', f'022_smooth_B7_seed{s}_step1000')
    man_path = os.path.join(res_dir, 'arm_manifest.json')
    man = json.load(open(man_path))

    def newest_step():
        cks = _glob.glob(os.path.join(out_dir, 'checkpoint-*'))
        return max((int(c.rsplit('-', 1)[1]) for c in cks), default=0)

    start_step = newest_step()
    for a in man.get('attempts', []):
        if 'ts_end' not in a:
            a.update({'interrupted': True, 'end_step': start_step,
                      'elapsed_sec': None, 'ts_end': 'runtime died before finally'})
    with open(man_path, 'w') as f:
        json.dump(man, f, indent=2)

    block_reason = blocked_resume_reason(man.get('attempts', []))
    assert block_reason is None, f'seed {s}: {block_reason}'
    active_workers = active_worker_processes(config_path)
    assert not active_workers, f'seed {s}: training worker already active: {active_workers}'

    final_artifacts = [os.path.join(out_dir, 'adapter_model.safetensors'),
                       os.path.join(out_dir, 'adapter_base.json'),
                       os.path.join(res_dir, 'grpo_trainlog.json')]
    if start_step >= 1000 and all(os.path.exists(f) for f in final_artifacts):
        print(f'seed {s}: complete, skipped')
        continue

    env = {'git': _git_info(), 'versions': _versions(),
           'resolved_dtype': resolve_dtype(cfg), 'ladder': 'disabled'}
    repo_top = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True,
                              text=True, check=True).stdout.strip()
    repo_prefix = os.path.relpath(os.getcwd(), repo_top)
    allowed_infra_files = {
        'notebooks/022_colab_b7_smooth1000.ipynb',
        'scripts/grpo_worker.py',
        'tests/test_7b_pilot.py',
        'tests/test_grpo_worker.py',
    }
    compatibility = man.setdefault('resume_compatibility', [])
    for a in man.get('attempts', []):
        prev = a.get('env')
        if prev:
            old_commit = prev['git'].get('commit')
            new_commit = env['git'].get('commit')
            if old_commit != new_commit:
                diff = subprocess.run(['git', 'diff', '--name-only', old_commit, new_commit],
                                      capture_output=True, text=True, check=True).stdout.splitlines()
                changed = []
                for name in diff:
                    prefix = repo_prefix + os.sep
                    changed.append(name[len(prefix):] if repo_prefix != '.' and name.startswith(prefix) else name)
                unexpected = sorted(set(changed) - allowed_infra_files)
                assert not unexpected, f'seed {s}: scientific code drifted between attempts: {unexpected}'
                row = {'from_commit': old_commit, 'to_commit': new_commit,
                       'changed_files': sorted(set(changed)),
                       'reason': 'live output infrastructure only'}
                if row not in compatibility:
                    compatibility.append(row)
                print(f'seed {s}: logging-only commit change verified')
            for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
                assert prev['versions'].get(lib) == env['versions'].get(lib), f'seed {s}: {lib} drifted'
            assert prev['resolved_dtype'] == env['resolved_dtype'], f'seed {s}: dtype drifted between attempts'
    attempt = {'start_step': start_step, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
               'gpu': gpu_snapshot(), 'env': env, 'completed': False}
    man.setdefault('attempts', []).append(attempt)
    with open(man_path, 'w') as f:
        json.dump(man, f, indent=2)

    worker_status_path = f'/content/022_grpo_worker_seed{s}.json'
    if os.path.exists(worker_status_path):
        os.remove(worker_status_path)
    worker_cmd = [sys.executable, '-u', 'scripts/grpo_worker.py',
                  '--config', config_path,
                  '--adapter', os.path.join(D, 'checkpoints', f'013_COLAB_FULL_B7_sft_grpo_seed{s}', 'sft'),
                  '--out-dir', out_dir,
                  '--log-dir', res_dir,
                  '--run-name', f'022_smooth_seed{s}',
                  '--status', worker_status_path,
                  '--report-to', 'wandb']
    t0 = time.time()
    proc = None
    parent_failure = None
    try:
        print(f'===== seed {s}, from step {start_step} =====')
        proc = stream_subprocess(worker_cmd)
        if proc.returncode != 0:
            raise RuntimeError(f'seed {s}: training worker failed with exit code {proc.returncode}')
    except BaseException as exc:
        parent_failure = {'error_type': type(exc).__name__,
                          'error_message': str(exc)[:2000]}
        raise
    finally:
        wall = time.time() - t0
        end_step = newest_step()
        worker_returncode = proc.returncode if proc is not None else None
        worker_status = json.load(open(worker_status_path)) if os.path.exists(worker_status_path) else {}
        worker_peak = worker_status.get('peak_vram_gib')
        error_type = worker_status.get('error_type') or (parent_failure or {}).get('error_type')
        error_message = worker_status.get('error_message') or (parent_failure or {}).get('error_message')
        manual_interrupt = error_type == 'KeyboardInterrupt'
        explicit_failure = ((worker_returncode not in (None, 0) or parent_failure is not None)
                            and not manual_interrupt)
        completed = (worker_status.get('completed') is True and end_step >= 1000
                     and all(os.path.exists(f) for f in final_artifacts))
        attempt.update({'end_step': end_step, 'ts_end': time.strftime('%Y-%m-%d %H:%M'),
                        'elapsed_sec': round(wall, 1), 'completed': completed,
                        'peak_vram_gib': worker_peak, 'worker_returncode': worker_returncode,
                        'error_type': error_type, 'error_message': error_message,
                        'cuda_oom': bool(worker_status.get('cuda_oom', False)),
                        'interrupted': manual_interrupt, 'resume_blocked': explicit_failure})
        with open(man_path, 'w') as f:
            json.dump(man, f, indent=2)
        gpu_row = gpu_snapshot()
        if worker_peak is not None:
            gpu_row['peak_vram_gb'] = worker_peak
        append_ledger(os.path.join(D, 'results', 'compute_ledger.jsonl'),
                      {'run_tag': f'022_smooth_B7_seed{s}', 'condition': 'SMOOTH_1000',
                       'timings_sec': {'grpo_train': round(wall, 1)},
                       'train_steps': end_step - start_step,
                       'completed': attempt['completed'], 'gpu': gpu_row})
    assert worker_status.get('completed') is True, f'seed {s}: worker status did not complete'
    assert attempt['completed'] is True, f'seed {s}: final training artifacts are incomplete'
    print(f'seed {s}: covered steps {start_step}-{end_step} in {wall/3600:.2f}h')


Streaming output truncated to the last 5000 lines.
│ │ letter of the │ are rarely    │               │              │           │ │
│ │ correct       │ involved in   │               │              │           │ │
│ │ option (A, B, │ the           │               │              │           │ │
│ │ C or D)       │ day-to-day    │               │              │           │ │
│ │ inside        │ operations.   │               │              │           │ │
│ │ <answer></an… │ Yes, some     │               │              │           │ │
│ │ tags.         │ council       │               │              │           │ │
│ │               │ representati… │               │              │           │ │
│ │ Executive     │ are appointed ��               │              │           │ │
│ │ directors are │ to operate as │               │              │           │ │
│ │ responsible   │ executives.<… │               │              │           │ │
│ │ for running   │ <answer>A</a… │               │      

In [ ]:
import json, os, glob
D = '/content/drive/MyDrive/inm363'
for s in SEEDS_TO_RUN:
    res_dir = os.path.join(D, 'results', f'022_smooth_B7_seed{s}_step1000')
    out_dir = os.path.join(D, 'checkpoints', f'022_smooth_B7_seed{s}', 'grpo')
    tl = json.load(open(os.path.join(res_dir, 'grpo_trainlog.json')))
    recs = [(e['step'], e) for e in tl if 'rewards/accuracy_reward/mean' in e and 'step' in e]
    from collections import Counter
    cc = Counter(st for st, _ in recs)
    assert all(v == 1 for v in cc.values()), f'seed {s}: duplicate steps {[k for k,v in cc.items() if v>1][:5]}'
    assert set(cc) == set(range(1, 1001)), f'seed {s}: reward records must cover 1..1000 exactly'

    lrs = [(st, e['learning_rate']) for st, e in sorted(recs) if 'learning_rate' in e]
    assert len(lrs) == 1000, f'seed {s}: expected 1000 lr records, found {len(lrs)}'
    assert 9.5e-6 <= lrs[0][1] <= 1.0e-5, f'seed {s}: first lr {lrs[0][1]} not at peak'
    assert lrs[-1][1] <= 2e-8, f'seed {s}: final lr {lrs[-1][1]} not near zero'
    assert all(v >= 0 for _, v in lrs), f'seed {s}: negative learning rate logged'
    for (s1, a), (s2, b) in zip(lrs, lrs[1:]):
        assert b <= a + 1e-12, f'seed {s}: lr increased between steps {s1} and {s2}'

    final_ck = sorted(glob.glob(os.path.join(out_dir, 'checkpoint-*')), key=lambda p: int(p.rsplit('-', 1)[1]))[-1]
    gs = json.load(open(os.path.join(final_ck, 'trainer_state.json')))['global_step']
    assert gs == 1000, f'seed {s}: final global_step {gs}'
    with open(os.path.join(res_dir, 'reward_curve.json'), 'w') as f:
        json.dump({str(st): e['rewards/accuracy_reward/mean'] for st, e in sorted(recs)}, f)
    print(f'seed {s}: gates passed, lr {lrs[0][1]:.2e} down to {lrs[-1][1]:.2e}, curve saved')
    # every registered grid checkpoint must have survived the retention limit
    for n in GRID[:-1]:
        ck = os.path.join(out_dir, f'checkpoint-{n}')
        assert os.path.isdir(ck), f'seed {s}: grid checkpoint {n} missing'
        gs_n = json.load(open(os.path.join(ck, 'trainer_state.json')))['global_step']
        assert gs_n == n, f'seed {s}: checkpoint-{n} says global_step {gs_n}'
    prov = json.load(open(os.path.join(out_dir, 'adapter_base.json')))
    assert prov.get('kind') == 'local', f'seed {s}: provenance kind {prov.get("kind")!r}'
    assert prov.get('value') == f'checkpoints/022_smooth_B7_seed{s}/sft_merged', \
        f'seed {s}: provenance points at {prov.get("value")!r}'
    print(f'seed {s}: grid checkpoints and provenance verified')

seed 2: gates passed, lr 1.00e-05 down to 1.00e-08, curve saved
seed 2: grid checkpoints and provenance verified


## 10. Endpoint and grid evaluations

The endpoint binds to the final adapter hash exactly as in the earlier arms.
Each grid point is evaluated from a local scratch copy: adapter files plus
the run's provenance file, source and copy hashes asserted equal, a grid
manifest written before evaluation so an old output can never be relabelled
as coming from this checkpoint.

In [ ]:
import hashlib, json, os
def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

D = '/content/drive/MyDrive/inm363'
import subprocess
free_mib = int(subprocess.check_output(
    ['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'],
    text=True).strip().splitlines()[0])
free_vram_gib = free_mib / 1024
print(f'free vram before evaluation: {free_vram_gib:.1f} GiB')
assert free_vram_gib >= 70, 'gpu memory is still occupied after training, restart the runtime and rerun'

NEEDED = ('grpo_eval.json', 'grpo_preds.jsonl', 'grpo_logprob.json', 'grpo_logprob_preds.jsonl')
for s in SEEDS_TO_RUN:
    out = os.path.join(D, 'results', f'022_smooth_B7_seed{s}_step1000')
    adapter = os.path.join(D, 'checkpoints', f'022_smooth_B7_seed{s}', 'grpo')
    binding = {'adapter_sha256': sha(os.path.join(adapter, 'adapter_model.safetensors')),
               'adapter_config_sha256': sha(os.path.join(adapter, 'adapter_config.json')),
               'provenance_sha256': sha(os.path.join(adapter, 'adapter_base.json'))}
    bind_path = os.path.join(out, 'eval_binding.json')
    have_any = any(os.path.exists(os.path.join(out, f)) for f in NEEDED)
    if have_any:
        # partial or complete outputs must provably belong to this adapter
        assert os.path.exists(bind_path), f'seed {s}: eval outputs without a binding, delete and rerun'
        assert json.load(open(bind_path)) == binding, f'seed {s}: outputs from a different adapter'
    else:
        with open(bind_path, 'w') as f:
            json.dump(binding, f)
    man_path = os.path.join(out, 'grpo_eval_manifest.json')
    complete = (all(os.path.exists(os.path.join(out, f)) for f in NEEDED)
                and os.path.exists(man_path) and json.load(open(man_path)).get('completed') is True)
    if complete:
        print(f'seed {s}: evaluation complete and bound, reverifying counts')
    else:
        # the evaluator resumes internally: a finished stage skips, a missing one runs
        print(f'===== eval seed {s} =====')
        !python scripts/eval_adapter.py --config configs/022_smooth_colab_b7_seed{s}.yaml --adapter {adapter} --condition grpo --out {out} --seed {s}
        if _exit_code != 0:
            raise SystemExit(f'seed {s} eval failed with exit code {_exit_code}')
    # both paths land here: a skipped arm gets the same validation as a fresh one
    ev = json.load(open(os.path.join(out, 'grpo_eval.json')))
    lp = json.load(open(os.path.join(out, 'grpo_logprob.json')))
    assert ev['n_items'] == 1884 and lp['n_items'] == 1884, f'seed {s}: item counts wrong'
    for pf in ('grpo_preds.jsonl', 'grpo_logprob_preds.jsonl'):
        with open(os.path.join(out, pf)) as fh:
            nrows = sum(1 for line in fh if line.strip())
        assert nrows == 1884, f'seed {s}: {pf} has {nrows} rows'

# the grid points, scratch-copied and hash-bound, one directory per step
import shutil
for s in SEEDS_TO_RUN:
    run = f'022_smooth_B7_seed{s}'
    ck_root = os.path.join(D, 'checkpoints', run, 'grpo')
    res_root = os.path.join(D, 'results', f'{run}_step1000')
    tl_sha = sha(os.path.join(res_root, 'grpo_trainlog.json'))
    for n in GRID[:-1]:
        tag = f'{run}_step{n}'
        out = os.path.join(D, 'results', tag)
        ck = os.path.join(ck_root, f'checkpoint-{n}')
        man_path = os.path.join(out, 'grid_manifest.json')
        print(f'===== {tag} =====')

        # local scratch, rebuilt from zero on every pass so leftovers are never trusted
        scratch = f'/content/022_scratch/seed{s}_step{n}'
        if os.path.isdir(scratch):
            shutil.rmtree(scratch)
        os.makedirs(scratch)
        hashes = {}
        for src_dir, fname, key in ((ck, 'adapter_model.safetensors', 'adapter'),
                                    (ck, 'adapter_config.json', 'adapter_config'),
                                    (ck_root, 'adapter_base.json', 'provenance')):
            a, b = os.path.join(src_dir, fname), os.path.join(scratch, fname)
            shutil.copy2(a, b)
            hashes[key + '_source'] = sha(a)
            hashes[key + '_copy'] = sha(b)
            assert hashes[key + '_source'] == hashes[key + '_copy'], f'{tag}: {fname} copy hash differs'

        expected = {'source_run': run, 'grid_step': n,
                    'trainer_state_global_step': json.load(open(os.path.join(ck, 'trainer_state.json')))['global_step'],
                    'hashes': hashes, 'trainlog_sha256': tl_sha}
        assert expected['trainer_state_global_step'] == n, f'{tag}: trainer state disagrees with the step'
        if any(os.path.exists(os.path.join(out, f)) for f in NEEDED):
            assert os.path.exists(man_path), f'{tag}: outputs exist without a grid manifest, delete {out} and rerun'
            existing = json.load(open(man_path))
            for k in ('source_run', 'grid_step', 'hashes', 'trainlog_sha256', 'trainer_state_global_step'):
                assert existing.get(k) == expected[k], f'{tag}: manifest field {k} does not match, refusing to relabel'
            print('existing outputs match the expected manifest, evaluator will skip')
        else:
            os.makedirs(out, exist_ok=True)
            with open(man_path, 'w') as f:
                json.dump(expected, f, indent=2)

        !python scripts/eval_adapter.py --config configs/022_smooth_colab_b7_seed{s}.yaml --adapter {scratch} --condition grpo --out {out} --seed {s}
        if _exit_code != 0:
            raise SystemExit(f'{tag} failed with exit code {_exit_code}')
        ev = json.load(open(os.path.join(out, 'grpo_eval.json')))
        lp = json.load(open(os.path.join(out, 'grpo_logprob.json')))
        assert ev['n_items'] == 1884 and lp['n_items'] == 1884, f'{tag}: item counts wrong'
        for pf in ('grpo_preds.jsonl', 'grpo_logprob_preds.jsonl'):
            with open(os.path.join(out, pf)) as fh:
                nrows = sum(1 for line in fh if line.strip())
            assert nrows == 1884, f'{tag}: {pf} has {nrows} rows'
        em = json.load(open(os.path.join(out, 'grpo_eval_manifest.json')))
        assert em.get('completed') is True, f'{tag}: eval manifest not completed'
        shutil.rmtree(scratch)
        print(f'{tag} done and verified, scratch removed')

free vram before evaluation: 79.2 GiB
===== eval seed 2 =====
Loading weights: 100% 339/339 [00:00<00:00, 2520.30it/s]
grpo eval (1884 items): 100% 118/118 [29:28<00:00, 14.99s/it, acc 0.798 fmt 0.990]
Loading weights: 100% 339/339 [00:00<00:00, 2553.38it/s]
grpo logprob (1884 items): 100% 118/118 [00:26<00:00,  4.41it/s, acc 0.799]
gen per-tier: {'easy': 0.8839590443686007, 'medium': 0.7914614121510674, 'hard': 0.6161616161616161}
probe per-tier: {'easy': 0.8976109215017065, 'medium': 0.8111658456486043, 'hard': 0.5631313131313131}
===== 022_smooth_B7_seed2_step200 =====
Loading weights: 100% 339/339 [00:00<00:00, 2858.22it/s]
grpo eval (1884 items): 100% 118/118 [29:07<00:00, 14.81s/it, acc 0.803 fmt 0.986]
Loading weights: 100% 339/339 [00:00<00:00, 2655.14it/s]
grpo logprob (1884 items): 100% 118/118 [00:26<00:00,  4.40it/s, acc 0.798]
gen per-tier: {'easy': 0.8930602957906713, 'medium': 0.7783251231527094, 'hard': 0.6388888888888888}
probe per-tier: {'easy': 0.8976109215017065, 'm

## 11. Registered comparison

Four fields per arm: the within-run step-500 versus step-1000
primary with Holm across arms once all three exist, the probe primary against
the pre-GRPO SFT probe with the interval rule per seed, the hard-tier margin
trajectory, and the descriptive schedule comparison against the 013 endpoint.
A pilot-state pass prints provisional numbers with no registered category.

In [ ]:
import json, os, sys
sys.path.insert(0, os.getcwd())
from analysis.stats import _read_preds, align_pred_rows, mcnemar_exact, paired_delta_ci, holm
from data.subjects import tier_of, TIERS

D = '/content/drive/MyDrive/inm363'
def preds_at(s, n):
    # step 1000 lives in the arm root, earlier grid points in their own dirs
    tag = f'022_smooth_B7_seed{s}_step1000' if n == 1000 else f'022_smooth_B7_seed{s}_step{n}'
    return _read_preds(os.path.join(D, 'results', tag, 'grpo_preds.jsonl'))

def probe_at(s, n):
    tag = f'022_smooth_B7_seed{s}_step1000' if n == 1000 else f'022_smooth_B7_seed{s}_step{n}'
    return _read_preds(os.path.join(D, 'results', tag, 'grpo_logprob_preds.jsonl'))

def acc(rows):
    return sum(r['correct'] for r in rows) / len(rows)

def hard_acc(rows):
    xs = [r['correct'] for r in rows if tier_of(r['subject']) == 'hard']
    return sum(xs) / len(xs)

NEEDED = ('grpo_eval.json', 'grpo_preds.jsonl', 'grpo_logprob.json', 'grpo_logprob_preds.jsonl')
done = [s for s in (0, 1, 2)
        if all(os.path.exists(os.path.join(D, 'results',
               f'022_smooth_B7_seed{s}_step1000' if n == 1000 else f'022_smooth_B7_seed{s}_step{n}', f))
               for n in GRID for f in NEEDED)]
print('arms with a complete grid:', done)
binding = len(done) == 3
if not binding:
    print('PROVISIONAL: no registered category until all three arms are complete')

rows, praw = [], []
for s in done:
    p500, p1000 = preds_at(s, 500), preds_at(s, 1000)
    a5, a10 = align_pred_rows(p1000, p500)
    c10, c5 = [r['correct'] for r in a5], [r['correct'] for r in a10]
    n01 = sum(1 for x, y in zip(c10, c5) if not x and y)
    n10_ = sum(1 for x, y in zip(c10, c5) if x and not y)
    d, lo, hi = paired_delta_ci(c5, c10)
    tiers = {}
    for t in TIERS:
        xs = [(x['correct'], y['correct']) for x, y in zip(a5, a10) if tier_of(x['subject']) == t]
        tiers[t] = (sum(a for a, _ in xs) / len(xs), sum(b for _, b in xs) / len(xs))

    # probe primary: endpoint probe against the pre-grpo sft probe, interval rule
    sft_lp = _read_preds(os.path.join(D, 'results', f'013_COLAB_FULL_B7_sft_grpo_seed{s}', 'sft_logprob_preds.jsonl'))
    ep_lp = probe_at(s, 1000)
    pa, pb = align_pred_rows(ep_lp, sft_lp)
    pd_, plo, phi = paired_delta_ci([r['correct'] for r in pb], [r['correct'] for r in pa])
    # failing the interval rule shows equivalence was not established, it does not show movement
    probe_class = 'unchanged (equivalence rule met)' if plo > -0.02 and phi < 0.02 else 'equivalence not established'

    # trajectory: overall, probe, hard generative, hard probe and the margin at
    # every grid point, with step 0 read from the recorded 013 sft baselines
    sft_g = _read_preds(os.path.join(D, 'results', f'013_COLAB_FULL_B7_sft_grpo_seed{s}', 'sft_preds.jsonl'))
    traj = {0: {'acc': acc(sft_g), 'probe_acc': acc(sft_lp), 'hard_gen': hard_acc(sft_g),
                'hard_probe': hard_acc(sft_lp), 'hard_margin': hard_acc(sft_g) - hard_acc(sft_lp)}}
    for n in GRID:
        g, p = preds_at(s, n), probe_at(s, n)
        traj[n] = {'acc': acc(g), 'probe_acc': acc(p), 'hard_gen': hard_acc(g), 'hard_probe': hard_acc(p),
                   'hard_margin': hard_acc(g) - hard_acc(p)}

    # descriptive schedule comparison: the 013 endpoint against this run at step 500
    orig = _read_preds(os.path.join(D, 'results', f'013_COLAB_FULL_B7_sft_grpo_seed{s}', 'grpo_preds.jsonl'))
    oa, ob = align_pred_rows(p500, orig)
    d_sched = acc(oa) - acc(ob)

    rows.append({'seed': s, 'acc_500': acc(p500), 'acc_1000': acc(p1000),
                 'within_run_delta': d, 'ci': (lo, hi), 'p_raw': mcnemar_exact(n01, n10_),
                 'tiers_1000_vs_500': tiers, 'probe_delta': pd_, 'probe_ci': (plo, phi),
                 'probe_class': probe_class, 'trajectory': traj,
                 'sched_500_vs_013': d_sched})
    praw.append(rows[-1]['p_raw'])

summary = {'binding': binding, 'common_within_run_reading': None, 'common_probe_reading': None, 'arms': []}
cats = []
adj = holm(praw) if binding else [None] * len(praw)
for r, pa_ in zip(rows, adj):
    if binding:
        material = abs(r['within_run_delta']) >= 0.02 and pa_ < 0.05
        if material:
            cat = 'material change' if r['within_run_delta'] > 0 else 'material decline'
        elif r['ci'][0] > -0.02 and r['ci'][1] < 0.02:
            cat = 'equivalent'
        else:
            cat = 'inconclusive'
        cats.append(cat)
    else:
        cat = 'PROVISIONAL, category deferred'
    r['category'] = cat
    r['holm_p'] = pa_
    print(f"seed {r['seed']}: step500 {r['acc_500']:.3f} -> step1000 {r['acc_1000']:.3f} "
          f"({r['within_run_delta']:+.3f}, raw p {r['p_raw']:.3g}, ci [{r['ci'][0]:+.3f},{r['ci'][1]:+.3f}]) -> {cat}")
    for t, (a, b) in r['tiers_1000_vs_500'].items():
        print(f"   {t}: step1000 {a:.3f} vs step500 {b:.3f}")
    print(f"   probe vs pre-grpo sft {r['probe_delta']:+.3f} ci [{r['probe_ci'][0]:+.3f},{r['probe_ci'][1]:+.3f}] -> {r['probe_class']}")
    for n in (0,) + GRID:
        tr = r['trajectory'][n]
        print(f"   step {n:4d}: acc {tr['acc']:.3f}  probe {tr['probe_acc']:.3f}  hard {tr['hard_gen']:.3f}  probe-hard {tr['hard_probe']:.3f}  margin {tr['hard_margin']*100:+.1f}pt")
    print(f"   schedule comparison, this run at 500 vs the 013 endpoint: {r['sched_500_vs_013']*100:+.1f}pt (descriptive)")
    summary['arms'].append(r)

if binding:
    if len(set(cats)) == 1 and cats[0] in ('equivalent', 'material change', 'material decline'):
        reading = {'equivalent': 'the second 500 steps left endpoints unchanged for these seeds',
                   'material change': 'the second 500 steps materially improved endpoints for these seeds',
                   'material decline': 'the second 500 steps materially degraded endpoints for these seeds'}[cats[0]]
    else:
        reading = 'no common within-run conclusion, mixed or inconclusive categories, reported arm by arm'
    summary['common_within_run_reading'] = reading
    print('\nWITHIN-RUN READING:', reading)
    pclasses = [r2['probe_class'] for r2 in rows]
    if len(set(pclasses)) == 1:
        pread = ('the probe is unchanged at double budget in all three arms'
                 if pclasses[0].startswith('unchanged')
                 else 'probe equivalence was not established in any arm')
    else:
        pread = 'no common probe conclusion, classifications differ, reported arm by arm'
    summary['common_probe_reading'] = pread
    print('PROBE READING:', pread)

with open(os.path.join(D, 'results', '022_comparison_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('summary saved')

arms with a complete grid: [0, 1, 2]
seed 0: step500 0.807 -> step1000 0.808 (+0.001, raw p 0.916, ci [-0.008,+0.011]) -> equivalent
   easy: step1000 0.892 vs step500 0.891
   medium: step1000 0.795 vs step500 0.777
   hard: step1000 0.641 vs step500 0.667
   probe vs pre-grpo sft +0.002 ci [-0.003,+0.007] -> unchanged (equivalence rule met)
   step    0: acc 0.780  probe 0.800  hard 0.588  probe-hard 0.568  margin +2.0pt
   step  200: acc 0.812  probe 0.800  hard 0.679  probe-hard 0.561  margin +11.9pt
   step  400: acc 0.808  probe 0.801  hard 0.654  probe-hard 0.561  margin +9.3pt
   step  500: acc 0.807  probe 0.802  hard 0.667  probe-hard 0.566  margin +10.1pt
   step  600: acc 0.805  probe 0.801  hard 0.646  probe-hard 0.563  margin +8.3pt
   step  800: acc 0.806  probe 0.799  hard 0.639  probe-hard 0.558  margin +8.1pt
   step 1000: acc 0.808  probe 0.803  hard 0.641  probe-hard 0.566  margin +7.6pt
   schedule comparison, this run at 500 vs the 013 endpoint: +0.5pt (descriptiv

## 12. Close the units record

In [ ]:
# closing balance, completes the open record for this session
import json, os, time
#cu_after = 254.12    # fill
# cu_after = 157.37     # fill from the resources panel again
cu_after = 93.1     # fill after the last seed!
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
